In [13]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path("../")
LABELSTUDIO_PATH = BASE_DIR / "data" / "labelstudio_output_fixed.json"
METADATA_PATH = BASE_DIR / "metadata_sample" / "sample_metadata_augmented_fixed.jsonl"

LABEL_FIELDS = [
    "jenis_atap_terluas",
    "jenis_dinding_terluas",
    "jenis_lantai_terluas",
]

ACTUAL_FIELDS = ["atap", "dinding", "lantai"]

print("Label Studio path :", LABELSTUDIO_PATH.resolve())
print("Metadata path     :", METADATA_PATH.resolve())

Label Studio path : C:\Users\Lutfi\Documents\Project\AITF\rutilahu-vlm-etl\data\labelstudio_output_fixed.json
Metadata path     : C:\Users\Lutfi\Documents\Project\AITF\rutilahu-vlm-etl\metadata_sample\sample_metadata_augmented_fixed.jsonl


In [14]:
def load_labelstudio_json(path: Path) -> pd.DataFrame:
    """Load labelstudio_output.json and return a tidy DataFrame.

    Supports:
    - a list of dicts
    - a dict with a top-level key containing records
    - a single dict record
    """
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    if isinstance(raw, list):
        records = raw
    elif isinstance(raw, dict):
        # Try common container keys first.
        for key in ["data", "results", "annotations", "items", "records"]:
            if key in raw and isinstance(raw[key], list):
                records = raw[key]
                break
        else:
            # If it looks like a single record, wrap it.
            records = [raw]
    else:
        raise ValueError(f"Unsupported JSON structure in {path}")

    df = pd.json_normalize(records)
    return df


def load_metadata_jsonl(path: Path) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    return df


def normalize_label(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        v = value.strip()
        return v if v else None
    return str(value).strip()


def tidy_labelstudio(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["house_id"] + [c for c in LABEL_FIELDS if c in df.columns]
    missing = [c for c in LABEL_FIELDS if c not in df.columns]
    if missing:
        print("Peringatan: kolom ini tidak ditemukan di labelstudio_output.json:", missing)
    out = df[cols].copy()
    for c in LABEL_FIELDS:
        if c in out.columns:
            out[c] = out[c].map(normalize_label)
    return out


def tidy_metadata(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "actual_label" in out.columns:
        actual = pd.json_normalize(out["actual_label"])
        actual.columns = [f"actual_{c}" for c in actual.columns]
        out = pd.concat([out.drop(columns=["actual_label"]), actual], axis=1)

    rename_map = {
        "actual_atap": "atap",
        "actual_dinding": "dinding",
        "actual_lantai": "lantai",
    }
    out = out.rename(columns=rename_map)

    keep_cols = ["house_id"] + [c for c in ACTUAL_FIELDS if c in out.columns]
    out = out[keep_cols].copy()
    for c in ACTUAL_FIELDS:
        if c in out.columns:
            out[c] = out[c].map(normalize_label)
    return out

In [15]:
# Load data
df_ls_raw = load_labelstudio_json(LABELSTUDIO_PATH)
df_md_raw = load_metadata_jsonl(METADATA_PATH)

print("Label Studio rows :", len(df_ls_raw))
print("Metadata rows     :", len(df_md_raw))

df_ls = tidy_labelstudio(df_ls_raw)
df_md = tidy_metadata(df_md_raw)

display(df_ls.head())
display(df_md.head())

Label Studio rows : 6321
Metadata rows     : 20118


,house_id,jenis_atap_terluas,jenis_dinding_terluas,jenis_lantai_terluas
0,H00002_EXT,Asbes,Tembok,Tidak terdeteksi
1,H00002_INT,Tidak terdeteksi,Tidak terdeteksi,Keramik
2,H00005_EXT,Genteng,Tembok,Tidak terdeteksi
3,H00005_INT,Tidak terdeteksi,Tidak terdeteksi,Keramik
4,H00009,Genteng,Tembok,Semen/bata merah


,house_id,atap,dinding,lantai
0,H00002_EXT,Genteng,Tembok,Tidak terdeteksi
1,H00002_INT,Tidak terdeteksi,Tidak terdeteksi,Ubin/tegel/teraso
2,H00005_EXT,Genteng,Tembok,Tidak terdeteksi
3,H00005_INT,Tidak terdeteksi,Tidak terdeteksi,Keramik
4,H00009,Genteng,Tembok,Semen/bata merah


In [16]:
# Unique labels from Label Studio
for col in LABEL_FIELDS:
    if col in df_ls.columns:
        print(f"\n=== Unique label: {col} ===")
        uniq = sorted(df_ls[col].dropna().unique().tolist())
        print(uniq)
        print("Count unique:", len(uniq))


=== Unique label: jenis_atap_terluas ===
['Asbes', 'Bambu', 'Beton', 'Genteng', 'Jerami/ijuk/daun-daunan/rumbia', 'Kayu/sirap', 'Lainnya', 'Seng', 'Tidak terdeteksi']
Count unique: 9

=== Unique label: jenis_dinding_terluas ===
['Anyaman bambu', 'Bambu', 'Batang kayu', 'Kayu/papan/gypsum/GRC/calciboard', 'Lainnya', 'Plesteran anyaman bambu/kawat', 'Tembok', 'Tidak terdeteksi']
Count unique: 8

=== Unique label: jenis_lantai_terluas ===
['Bambu', 'Kayu/papan', 'Keramik', 'Lainnya', 'Marmer/granit', 'Parket/vinil/karpet', 'Semen/bata merah', 'Tanah', 'Tidak terdeteksi', 'Ubin/tegel/teraso']
Count unique: 10


In [17]:
# Unique labels from actual_label (metadata)
for col in ACTUAL_FIELDS:
    if col in df_md.columns:
        print(f"\n=== Unique label: {col} ===")
        uniq = sorted(df_md[col].dropna().unique().tolist())
        print(uniq)
        print("Count unique:", len(uniq))


=== Unique label: atap ===
['Asbes', 'Bambu', 'Beton', 'Genteng', 'Jerami/ijuk/daun-daunan/rumbia', 'Kayu/sirap', 'Lainnya', 'Seng', 'Tidak terdeteksi', 'unclassified']
Count unique: 10

=== Unique label: dinding ===
['Anyaman bambu', 'Bambu', 'Batang kayu', 'Kayu/papan/gypsum/GRC/calciboard', 'Lainnya', 'Plesteran anyaman bambu/kawat', 'Tembok', 'Tidak terdeteksi', 'unclassified']
Count unique: 9

=== Unique label: lantai ===
['Bambu', 'Kayu/papan', 'Keramik', 'Lainnya', 'Marmer/granit', 'Parket/vinil/karpet', 'Semen/bata merah', 'Tanah', 'Tidak terdeteksi', 'Ubin/tegel/teraso', 'unclassified']
Count unique: 11


In [18]:
def value_counts_table(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    frames = []
    for col in cols:
        if col in df.columns:
            vc = df[col].value_counts(dropna=False).reset_index()
            vc.columns = [col, "count"]
            vc.insert(0, "field", col)
            frames.append(vc)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

print("=== Distribusi label Label Studio ===")
display(value_counts_table(df_ls, LABEL_FIELDS))

print("=== Distribusi label actual_label ===")
display(value_counts_table(df_md, ACTUAL_FIELDS))

=== Distribusi label Label Studio ===


,field,jenis_atap_terluas,count,jenis_dinding_terluas,jenis_lantai_terluas
0,jenis_atap_terluas,Tidak terdeteksi,2783,NaN,NaN
1,jenis_atap_terluas,Genteng,2242,NaN,NaN
2,jenis_atap_terluas,Asbes,783,NaN,NaN
3,jenis_atap_terluas,Seng,354,NaN,NaN
4,jenis_atap_terluas,Beton,133,NaN,NaN
5,jenis_atap_terluas,Jerami/ijuk/daun-daunan/rumbia,13,NaN,NaN
6,jenis_atap_terluas,Kayu/sirap,8,NaN,NaN
7,jenis_atap_terluas,Lainnya,3,NaN,NaN
8,jenis_atap_terluas,Bambu,2,NaN,NaN
9,jenis_dinding_terluas,NaN,4499,Tembok,NaN


=== Distribusi label actual_label ===


,field,atap,count,dinding,lantai
0,atap,Genteng,13079,NaN,NaN
1,atap,Tidak terdeteksi,4058,NaN,NaN
2,atap,Asbes,1729,NaN,NaN
3,atap,Seng,599,NaN,NaN
4,atap,Beton,421,NaN,NaN
5,atap,unclassified,93,NaN,NaN
6,atap,Lainnya,83,NaN,NaN
7,atap,Bambu,30,NaN,NaN
8,atap,Jerami/ijuk/daun-daunan/rumbia,14,NaN,NaN
9,atap,Kayu/sirap,12,NaN,NaN
